# Notebook 03 — Strategy 2: Earnings Spike

**Author:** Maxandre Goillot  
**Result:** Sharpe **+0.46** (N=15 events) ⚠️ Inconclusive

This notebook covers:
1. Earnings event detection from Polymarket volume spikes
2. IC (Information Coefficient) analysis — pre vs. post announcement
3. Placebo test: is the signal specific to announcement windows?
4. Backtest results
5. Version history (V0.1 → V0.6)


In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

ROOT = Path("..")
sys.path.insert(0, str(ROOT))

from backtest.engine import (
    load_sp100_bars, load_sp100_returns, load_polymarket_features,
    load_quotes, apply_epsilon_delay, compute_spread_costs,
)
from backtest.metrics import compute_pnl, buy_and_hold_benchmark
from strategies.s2_earnings_spike.strategy import (
    _identify_earnings_events, _compute_ic, generate_signals,
)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Setup OK")

## 1. Event Detection

In [ ]:
df_poly    = load_polymarket_features(split="full", resample="15min")
df_returns = load_sp100_returns(resample="15min")

events = _identify_earnings_events(df_poly)
print(f"Total earnings events detected: {len(events)}")

df_events = pd.DataFrame([
    {"timestamp": t, "category": cat[:40], "ticker": tick}
    for t, cat, tick in events
]).sort_values("timestamp")
df_events.head(20)

## 2. IC Analysis — Pre vs. Post Announcement

In [ ]:
ic_pre, ic_post = [], []
for t, cat, tick in events:
    # Post: compute IC at event time
    ic_p = _compute_ic(df_poly, cat, df_returns, tick, t, lag_bars=1)
    ic_post.append(ic_p)

    # Pre: shift 8 bars back (~2h)
    if t not in df_poly.index:
        ic_pre.append(np.nan)
        continue
    idx  = df_poly.index.get_loc(t)
    pre_t = df_poly.index[max(0, idx - 8)]
    ic_pre.append(_compute_ic(df_poly, cat, df_returns, tick, pre_t, lag_bars=1))

ic_pre  = np.array(ic_pre,  dtype=float)
ic_post = np.array(ic_post, dtype=float)

mask = ~np.isnan(ic_pre) & ~np.isnan(ic_post)
print(f"Valid events: {mask.sum()}")
print(f"IC pre-announcement:   mean={ic_pre[mask].mean():+.3f}  std={ic_pre[mask].std():.3f}")
print(f"IC post-announcement:  mean={ic_post[mask].mean():+.3f}  std={ic_post[mask].std():.3f}")

t_stat, p_val = stats.ttest_ind(ic_post[mask], ic_pre[mask], alternative="greater")
print(f"t-stat={t_stat:.3f}  p-value={p_val:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(-1, 1, 20)
ax.hist(ic_pre[mask],  bins=bins, alpha=0.6, label=f"Pre-announcement  (μ={ic_pre[mask].mean():+.2f})",  color="#94A3B8")
ax.hist(ic_post[mask], bins=bins, alpha=0.8, label=f"Post-announcement (μ={ic_post[mask].mean():+.2f})", color="#3B82F6")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Information Coefficient (IC)")
ax.set_ylabel("Count")
ax.set_title(f"S2 — IC Distribution  |  t={t_stat:.2f}, p={p_val:.3f}", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Placebo Analysis

In [ ]:
placebo_path = ROOT / "strategies" / "s2_earnings_spike" / "placebo_results" / "placebo_summary.json"
if placebo_path.exists():
    with open(placebo_path) as f:
        p = json.load(f)
    print(f"Real IC mean:    {p['real_mean']:+.3f}  (n={p['n_events']})")
    print(f"Placebo IC mean: {p['placebo_mean']:+.3f}  (n={p['n_placebo']})")
    print(f"t-stat:   {p['t_stat']:+.3f}")
    print(f"p-value:  {p['p_value']:.4f}")
    verdict = "SPECIFIC to announcement windows" if p["p_value"] < 0.05 else "INCONCLUSIVE"
    print(f"Verdict:  {verdict}")
else:
    print("Placebo results not found. Run: python strategies/s2_earnings_spike/placebo_analysis.py")

## 4. Backtest — Test Set (Nov–Dec 2025)

In [ ]:
df_bars_test   = load_sp100_bars(resample="15min", start="2025-11-01")
df_returns_test = df_bars_test.pct_change(1, fill_method=None)
df_poly_test    = load_polymarket_features(split="test", resample="15min")
df_quotes       = load_quotes()

raw_weights = generate_signals(df_poly_test, df_bars_test)
delayed     = apply_epsilon_delay(raw_weights, epsilon_minutes=1)
costs       = compute_spread_costs(delayed, df_quotes)
result      = compute_pnl(delayed, df_returns_test, costs, is_capacity_constrained=False)

m = result["metrics"]
bh = buy_and_hold_benchmark(df_returns_test)

print(f"Sharpe:       {m['sharpe_ratio']:+.3f}")
print(f"Net return:   {m['net_cumulative_return']*100:+.2f}%")
print(f"Max DD:       {m['max_drawdown']*100:.2f}%")
print(f"Trades:       {m['n_trades']}")
print(f"B&H Sharpe:   {bh['metrics']['sharpe_ratio']:+.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(result["cum_net"].index, result["cum_net"].values * 100,
        label=f"S2 Net ({result['cum_net'].iloc[-1]*100:+.2f}%)",
        color="#10B981", lw=2.5)
ax.plot(bh["cum_net"].index, bh["cum_net"].values * 100,
        label=f"S&P 100 B&H ({bh['cum_net'].iloc[-1]*100:+.2f}%)",
        color="#64748B", lw=2.0, ls=":")
ax.axhline(0, color="grey", lw=0.8, ls="--")
ax.set_title(f"S2 — Earnings Spike  |  Sharpe={m['sharpe_ratio']:+.3f}", fontsize=13, fontweight="bold")
ax.set_ylabel("Cumulative Return (%)")
ax.legend()
ax.grid(True, ls=":", alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Strategy Version History

| Version | Key change | Sharpe |
|---------|-----------|:------:|
| V0.1 | Naive IC filter (threshold 0.5) | +0.12 |
| V0.2 | Added volume spike event detection (Z > 3) | +0.21 |
| V0.3 | Asymmetric pre/post windows (1h pre, 2h post) | +0.35 |
| V0.4 | Placebo test confirms window specificity (p=0.031) | +0.35 |
| V0.5 | Mean-reversion direction (short after bullish spike) | +0.41 |
| **V0.6** | **Holding period sweep → optimum at 8 bars (2h)** | **+0.46** |

**Conclusion:** The signal is real (p=0.031) but N=15 events is too small to be conclusive at 1% significance. Need 30+ events for a powered test.